RAG Pipeline to Vector DB Pipeline

In [11]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [30]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# ### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")    
    return all_documents

# Point to the 'pdf' folder inside your 'notebook' directory
all_pdf_documents = process_all_pdfs("./pdf")

C:\Users\Dell\AppData\Local\Temp\ipykernel_24776\1496005645.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Found 4 PDF files to process

Processing: Attention.pdf
  ✓ Loaded 2 pages

Processing: Embeddings.pdf
  ✓ Loaded 2 pages

Processing: Object_Detection.pdf
  ✓ Loaded 2 pages

Processing: Proposal (1).pdf
  ✓ Loaded 2 pages

Total documents loaded: 8


In [34]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-10T18:17:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-10T18:17:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdf\\Attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content="Attention\nHow Transformers Decide What to Focus On\n1. What Problem Attention Solves\nOlder sequence models (RNNs, LSTMs) compress an entire input sequence into a single fixed-size hidden\nvector before producing output, creating a bottleneck for long sequences. Attention lets a model look back at\nall input positions at every output step and decide, dynamically, which ones matter most right now.\n2. The Core Idea: Query, Key, Value\nEvery attention mechanism reduces to comparing a query (what am I looking for) against a set of keys\n(what each po

In [40]:
###text splitting into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_document(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...")
    print(f"Metadata:{split_docs[0].metadata}")

    return split_docs

In [47]:
chunks=split_document(all_pdf_documents)
chunks


Example chunk:
Content: Attention
How Transformers Decide What to Focus On
1. What Problem Attention Solves
Older sequence models (RNNs, LSTMs) compress an entire input sequence into a single fixed-size hidden
vector before ...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-10T18:17:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-10T18:17:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdf\\Attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-10T18:17:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-10T18:17:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdf\\Attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Attention\nHow Transformers Decide What to Focus On\n1. What Problem Attention Solves\nOlder sequence models (RNNs, LSTMs) compress an entire input sequence into a single fixed-size hidden\nvector before producing output, creating a bottleneck for long sequences. Attention lets a model look back at\nall input positions at every output step and decide, dynamically, which ones matter most right now.\n2. The Core Idea: Query, Key, Value\nEvery attention mechanism reduces to comparing a query (what am I looking for) against a set of keys\n(what each po

embeding and VectorstoreDB

In [48]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# Initialize embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3682.41it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Dell\AppData\Local\Temp\ipykernel_24776\389698084.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


vectorstore

In [44]:
import os
import chromadb

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Doc content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist() if isinstance(embedding, np.ndarray) else embedding)

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store!")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [49]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-10T18:17:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-10T18:17:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'pdf\\Attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Attention\nHow Transformers Decide What to Focus On\n1. What Problem Attention Solves\nOlder sequence models (RNNs, LSTMs) compress an entire input sequence into a single fixed-size hidden\nvector before producing output, creating a bottleneck for long sequences. Attention lets a model look back at\nall input positions at every output step and decide, dynamically, which ones matter most right now.\n2. The Core Idea: Query, Key, Value\nEvery attention mechanism reduces to comparing a query (what am I looking for) against a set of keys\n(what each po

In [53]:
###convert text to embeddings

texts=[doc.page_content for doc in chunks]
texts

##generate embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 19 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

Generated embeddings with shape: (19, 384)
Adding 19 documents to vector store...
Successfully added 19 documents to vector store!
Total documents in collection: 19


Retrieval Pipeline From VectorStore

In [58]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
             
             print(f"Error during retrieval: {e}")
        return []
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [59]:
rag_retriever.retrieve("what is attention")

Retrieving documents for query: 'what is attention'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [60]:
rag_retriever.retrieve("unified multi-task learning framework")

Retrieving documents for query: 'unified multi-task learning framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

Integration Vectordb Context pipeline with LLM output

In [61]:
###simple RAG pipeline with Groq LLM
if groq_LLm:
    query="What is unified Multi-Task learning framework?"
    retrieved_docs=rag_retriever.retrieve(query, top_k=3, score_threshold=0.1)

    if retrieved_docs:
        #combine top retrieved documents as context
        combined_context="\n\n".join([doc['content']for doc in retrieved_docs])

        #generate response using groq llm
        response=groq_LLm.generate_response(query,combined_context)
        print(f"\nResponse:\n{response}")
    else:
        print("no relevant documents found for the query.")    

NameError: name 'groq_LLm' is not defined

In [89]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# 1. Load .env file from current or parent directory
current_dir = Path.cwd()
env_paths = [current_dir / ".env", current_dir.parent / ".env"]

env_loaded = False
for path in env_paths:
  if path.exists():
    load_dotenv(dotenv_path=path, override=True)
    env_loaded = True
    print(f"Loaded .env from: {path}")
    break

if not env_loaded:
  raise FileNotFoundError("Could not find a .env file!")

if not os.getenv("GROQ_API_KEY"):
  raise ValueError("GROQ_API_KEY is missing from your .env file!")

# 2. Initialize the Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
)


# 3. Simple RAG Function
def rag_simple(query, retriever, llm, top_k=3):
  # Retrieve the context
  results = retriever.retrieve(query, top_k=top_k)

  # Extract content into a single string
  context = (
      "\n\n".join([doc["content"] for doc in results]) if results else ""
  )

  if not context:
    return "No relevant context found to answer the question."

  # Generate answer using f-string prompt
  prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}
Answer:"""

  response = llm.invoke(prompt)
  return response.content


print("LLM and RAG function ready to go!")

Loaded .env from: c:\Users\Dell\Desktop\ytrag\notebook\.env
LLM and RAG function ready to go!


In [91]:
answer=rag_simple("What is embeddings?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is embeddings?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


An embedding is a mapping from a discrete or high-dimensional object to a dense vector of real numbers in a lower-dimensional continuous space, where geometric relationships reflect semantic or functional relationships in the original data.


Enchanced rag pipeline features

In [92]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(
    query, retriever, llm, top_k=5, min_score=0.2, return_context=False
):
  """RAG pipeline with extra features:

  - Returns answer, sources, confidence score, and optionally full context.
  """
  results = retriever.retrieve(
      query, top_k=top_k, score_threshold=min_score
  )
  if not results:
    return {
        'answer': 'No relevant context found.',
        'sources': [],
        'confidence': 0.0,
        'context': '',
    }

  # Prepare context and sources
  context = '\n\n'.join([doc['content'] for doc in results])
  sources = [
      {
          'source': doc['metadata'].get(
              'source_file', doc['metadata'].get('source', 'unknown')
          ),
          'page': doc['metadata'].get('page', 'unknown'),
          'score': doc['similarity_score'],
          'preview': doc['content'][:120] + '...',
      }
      for doc in results
  ]
  confidence = max([doc['similarity_score'] for doc in results])

  # Generate answer
  prompt = f'Use the following context to answer the question concisely.\nContext:\n{context}\nQuestion: {query}\nAnswer:'
  response = llm.invoke([prompt.format(context=context, query=query)])

  output = {
      'answer': response.content,
      'sources': sources,
      'confidence': confidence,
  }

  if return_context:
    output['context'] = context

  return output
# Example usage:
result = rag_advanced(
    'What is attention mechanism?',
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)
print('Answer:', result['answer'])
print('Sources:', result['sources'])
print('Confidence:', result['confidence'])
print('Context Preview:', result['context'][:300])



Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [93]:
import time
from typing import Any, Dict, List


# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
class AdvancedRAGPipeline:

  def __init__(self, retriever, llm):
    self.retriever = retriever
    self.llm = llm
    self.history = []  # Store query history

  def query(
      self,
      question: str,
      top_k: int = 5,
      min_score: float = 0.2,
      stream: bool = False,
      summarize: bool = False,
  ) -> Dict[str, Any]:
    # Retrieve relevant documents
    results = self.retriever.retrieve(
        question, top_k=top_k, score_threshold=min_score
    )

    if not results:
      answer = "No relevant context found."
      sources = []
      context = ""
    else:
      context = "\n\n".join([doc["content"] for doc in results])
      sources = [
          {
              "source": doc["metadata"].get(
                  "source_file", doc["metadata"].get("source", "unknown")
              ),
              "page": doc["metadata"].get("page", "unknown"),
              "score": doc["similarity_score"],
              "preview": doc["content"][:120] + "...",
          }
          for doc in results
      ]

    # Streaming answer simulation
    prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\nQuestion: {question}\nAnswer:"

    if stream:
      print("Streaming answer:")
      for i in range(0, len(prompt), 80):
        print(prompt[i : i + 80], end="", flush=True)
        time.sleep(0.05)
      print()

    response = self.llm.invoke([
        prompt.format(context=context, question=question)
    ])
    answer = response.content

    # Add citations to answer
    citations = [
        f"[{i+1}] {src['source']} (page {src['page']})"
        for i, src in enumerate(sources)
    ]
    answer_with_citations = (
        answer + "\n\nCitations:\n" + "\n".join(citations)
        if citations
        else answer
    )

    # Optionally summarize answer
    summary = None
    if summarize and answer:
      summary_prompt = (
          f"Summarize the following answer in 2 sentences:\n{answer}"
      )
      summary_resp = self.llm.invoke([summary_prompt])
      summary = summary_resp.content

    # Store query history
    self.history.append({
        "question": question,
        "answer": answer,
        "sources": sources,
        "summary": summary,
    })

    return {
        "question": question,
        "answer": answer_with_citations,
        "sources": sources,
        "summary": summary,
        "history": self.history,
    }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(
    "what is attention is all you need",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True,
)

print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:

Question: what is attention is all you need
Answer:



Final Answer: "Attention Is All You Need" refers to a 2017 research paper introducing the Transformer model, which relies solely on self-attention mechanisms for machine translation tasks, replacing traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectures.
Summary: The "Attention Is All You Need" research paper, published in 2017, introduced the Transformer model, a new approach to machine translation. The Transformer model relies on self-attention mechanisms, abandoning traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectures in favor of a more innovative approach.
History: {'question': 'what is attention is all you need', 'answer': '"Attention Is All You Need" refers to a 2017 research paper introducing the Transformer model, which relies solely on self-attention mechanisms for machine translation tasks, replacing traditional recurrent neural network (RNN) and convolutional neural network (CNN) architectur